# Sparse Autoencoder na embeddingach ResNet-18

Tutorial trwa około 30 minut. Kod opiera się na artykule *Learning Concept Bottleneck Models from Mechanistic Explanations* (De Santis et al., ICLR 2026).

W Google Colab włącz GPU w menu Środowisko wykonawcze, Zmień typ środowiska wykonawczego, T4 GPU. Na CPU ten notatnik także zadziała, ale ekstrakcja embeddingów potrwa od 3 do 5 minut zamiast 35 sekund.

Plan zajęć:
1. Wczytanie gotowego pliku `model.pt` i zbioru CUB-200-2011.
2. Ekstrakcja 512-wymiarowych embeddingów z warstwy `avgpool` i podgląd jednego zdjęcia.
3. Budowa i trening Sparse Autoencodera (SAE).
4. Porównanie embeddingu z ResNet-18 i z SAE dla tego samego zdjęcia.
5. Pomiar `recovered_loss`, `recovered_accuracy` oraz pruning neuronów SAE.
6. Przegląd galerii Top-K i analiza czystości klasowej.


In [ ]:
# Konfiguracja pobierania model.pt oraz hiperparametrów SAE
MODEL_PATH = "model.pt"
GDRIVE_ID = "1yWDCwEFmU0hRjS_GwPcCgGZ8_IuY5ip5"              # wklej tutaj ID pliku model.pt z Google Drive

# Hiperparametry SAE dobrane dla tego modelu ResNet-18
L1_COEFFICIENT = 0.008      # cel: L0 około 8 do 9 aktywnych neuronów na zdjęcie (w M-CBM: 7.66)
EXPANSION_FACTOR = 1        # 512 neuronów SAE na wyjściu enkodera
SAE_BATCH_SIZE = 64
SAE_LR = 1e-4
SAE_EPOCHS = 500
SAE_PATIENCE = 50
MIN_ACTIVE_IMAGES = 25      # minimalna liczba zdjęć train, na których neuron musi być aktywny
CE_DROP_TOLERANCE = 0.005   # dopuszczalny spadek recovered_loss podczas pruningu

SEED = 42
VAL_SPLIT = 0.1


In [ ]:
import copy
import math
import os
import random
import tarfile
import textwrap
import urllib.request
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import models, transforms
from tqdm.auto import tqdm


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 2 if os.path.exists("/content") else 0
print(f"Urządzenie: {device}. PyTorch: {torch.__version__}.")
if device.type == "cpu":
    print("Używasz CPU. W Google Colab przełącz środowisko na T4 GPU.")


## Krok 1. Pobranie `model.pt` i CUB-200-2011

Uzupełnij `GDRIVE_ID` w pierwszej komórce kodu. Kolejna komórka pobierze `model.pt` z Google Drive za pomocą `gdown`.


In [ ]:
model_file = Path(MODEL_PATH)

if not model_file.exists():
    if not GDRIVE_ID:
        raise ValueError("Ustaw GDRIVE_ID w pierwszej komórce kodu.")
    import gdown
    gdown.download(id=GDRIVE_ID, output=str(model_file), quiet=False)

assert model_file.exists(), f"Brak pliku {model_file}"
print(f"Checkpoint: {model_file.resolve()} ({model_file.stat().st_size / 1e6:.1f} MB)")


In [ ]:
# Pobranie i rozpakowanie CUB-200-2011 (1.1 GB)
DATASET_FOLDER = Path("datasets")
CUB_ROOT = DATASET_FOLDER / "CUB_200_2011"
CUB_URL = "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz?download=1"

if not (CUB_ROOT / "images.txt").exists():
    DATASET_FOLDER.mkdir(parents=True, exist_ok=True)
    archive = DATASET_FOLDER / "CUB_200_2011.tgz"
    if not archive.exists():
        req = urllib.request.Request(CUB_URL, headers={"User-Agent": "concept-learning-colab"})
        with urllib.request.urlopen(req) as resp, archive.open("wb") as f:
            total = int(resp.headers.get("Content-Length", 0)) or None
            with tqdm(total=total, unit="B", unit_scale=True, desc="CUB_200_2011.tgz") as bar:
                while chunk := resp.read(1 << 20):
                    f.write(chunk)
                    bar.update(len(chunk))
    print("Rozpakowuję archiwum.")
    with tarfile.open(archive) as tar:
        tar.extractall(DATASET_FOLDER)
    archive.unlink()

print(f"Zbiór CUB-200-2011 znajduje się w {CUB_ROOT.resolve()}.")


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class CUBDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, transform=None, train=True):
        self.root_dir = Path(root_dir)
        self.transform = transform

        images = pd.read_csv(self.root_dir / "images.txt", sep=" ", header=None, names=["img_id", "filepath"])
        labels = pd.read_csv(self.root_dir / "image_class_labels.txt", sep=" ", header=None, names=["img_id", "label"])
        split = pd.read_csv(self.root_dir / "train_test_split.txt", sep=" ", header=None, names=["img_id", "is_train"])
        classes = pd.read_csv(self.root_dir / "classes.txt", sep=" ", header=None, names=["class_id", "name"])

        data = images.merge(labels, on="img_id").merge(split, on="img_id")
        self.data = data[data["is_train"] == int(train)].reset_index(drop=True)
        self.targets = (self.data["label"] - 1).astype(int).tolist()
        self.class_names = [name.split(".", 1)[-1].replace("_", " ") for name in classes.sort_values("class_id")["name"]]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.load_image(idx)
        if self.transform:
            image = self.transform(image)
        return image, self.targets[idx]

    def get_image_name(self, idx):
        return str(self.data.iloc[idx]["filepath"])

    def load_image(self, idx):
        return self.load_image_by_name(self.get_image_name(idx))

    def load_image_by_name(self, image_name):
        return Image.open(self.root_dir / "images" / image_name).convert("RGB")


eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
display_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])

# Stratyfikowany podział oficjalnego splitu train (5994 zdjęcia) na 5394 train i 600 val
train_full = CUBDataset(CUB_ROOT, transform=eval_transform, train=True)
train_idx, val_idx = train_test_split(
    range(len(train_full.targets)), test_size=VAL_SPLIT, random_state=SEED, stratify=train_full.targets
)
train_idx, val_idx = sorted(train_idx), sorted(val_idx)
train_ds = Subset(train_full, train_idx)
val_ds = Subset(train_full, val_idx)
train_image_names = [train_full.get_image_name(i) for i in train_idx]

print(f"Liczba klas: {len(train_full.class_names)}. Train: {len(train_ds)}. Val: {len(val_ds)}.")


### Podział ResNet-18 na `feature_map` i `head`

Klasa `Backbone` dzieli ResNet-18 na dwie części zgodnie ze strukturą zapisaną w `model.pt`:
- `feature_map(x)` zwraca mapę cech z warstwy `layer4` o kształcie `(B, 512, 7, 7)`.
- `embed(x)` uśrednia przestrzennie wyjście `feature_map(x)` i zwraca 512-wymiarowy embedding `(B, 512)`.
- `head(embed)` to warstwa liniowa z 512 wymiarów na 200 klas.


In [ ]:
class Backbone(nn.Module):
    def __init__(self, feature_map, head):
        super().__init__()
        self.feature_map = feature_map
        self.head = head

    def embed(self, x):
        return self.feature_map(x).mean(dim=(2, 3))

    def forward(self, x):
        return self.head(self.embed(x))


def load_classifier(checkpoint_path, device):
    state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    resnet = models.resnet18(weights=None)
    resnet.fc = nn.Linear(resnet.fc.in_features, state["num_classes"])
    feature_map = nn.Sequential(
        resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
        resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4,
    )
    backbone = Backbone(feature_map, resnet.fc)
    backbone.load_state_dict(state["state_dict"])
    print(f"Wczytano model z epoki {state.get('epoch', '?')}. Zapisana dokładność val: {state.get('val_accuracy', 0):.2%}.")
    return backbone.to(device).eval()


backbone = load_classifier(model_file, device)

with torch.no_grad():
    dummy = torch.randn(1, 3, 224, 224, device=device)
    print("Kształt feature_map:", tuple(backbone.feature_map(dummy).shape))
    print("Kształt embed:      ", tuple(backbone.embed(dummy).shape))
    print("Kształt logits:     ", tuple(backbone(dummy).shape))


## Krok 2. Ekstrakcja 512-wymiarowych embeddingów dla `train` i `val`

Przepuszczamy zdjęcia przez zamrożony `backbone.embed(x)` z deterministyczną transformacją `eval_transform`. Pomijamy split `test`, ponieważ trening SAE, pruning i galeria Top-K korzystają wyłącznie ze splitów `train` i `val`. Na GPU T4 ekstrakcja zajmuje około 35 sekund.


In [ ]:
@torch.no_grad()
def extract_split(backbone, dataset, split_name, batch_size=128):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )
    feats, labs, correct = [], [], 0
    for images, batch_labels in tqdm(loader, desc=f"Ekstrakcja {split_name}", unit="batch"):
        emb = backbone.embed(images.to(device, non_blocking=True))
        preds = backbone.head(emb).argmax(1).cpu()
        correct += (preds == batch_labels).sum().item()
        feats.append(emb.float().cpu())
        labs.append(batch_labels)
    features = torch.cat(feats)
    labels = torch.cat(labs)
    return features, labels, correct / len(dataset)


train_features, train_labels, train_acc = extract_split(backbone, train_ds, "train")
val_features, val_labels, val_acc = extract_split(backbone, val_ds, "val")

print(f"Train: {tuple(train_features.shape)}, dokładność klasyfikatora: {train_acc:.2%}.")
print(f"Val:   {tuple(val_features.shape)}, dokładność klasyfikatora: {val_acc:.2%}.")
print(f"Średnia norma L2 embeddingów train: {train_features.norm(dim=1).mean():.2f}.")
print("Model autorów M-CBM ma normę L2 około 11.25. Dlatego tutaj używamy L1 = 0.008 zamiast 0.002.")


### Podgląd surowego embeddingu z ResNet-18

Spójrz na zdjęcie o indeksie `SAMPLE_IDX` i jego 512-wymiarowy embedding po warstwie `avgpool`. Niemal wszystkie 512 wymiarów mają wartości większe od zera. Po wytrenowaniu SAE porównamy ten sam obraz z jego rzadką reprezentacją.


In [ ]:
SAMPLE_IDX = 0  # dowolny indeks od 0 do 5393

sample_img = train_full.load_image_by_name(train_image_names[SAMPLE_IDX])
sample_label = train_full.class_names[int(train_labels[SAMPLE_IDX])]
sample_resnet_emb = train_features[SAMPLE_IDX].numpy()
n_active_resnet = int((sample_resnet_emb > 0).sum())

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1, 2.4]})
axes[0].imshow(display_transform(sample_img))
axes[0].set_title(f"Zdjęcie #{SAMPLE_IDX}:\n{sample_label}", fontsize=11)
axes[0].axis("off")

axes[1].bar(np.arange(len(sample_resnet_emb)), sample_resnet_emb, width=1.0, color="#1d4e89")
axes[1].set_xlim(0, len(sample_resnet_emb))
axes[1].set_xlabel("Indeks wymiaru embeddingu (0 do 511)")
axes[1].set_ylabel("Wartość po avgpool")
axes[1].set_title(f"Embedding ResNet-18: aktywnych {n_active_resnet} z 512 wymiarów ({n_active_resnet / 512:.1%})")
axes[1].grid(True, axis="y", alpha=0.4)

plt.tight_layout()
plt.show()


## Krok 3. Architektura Sparse Autoencodera i funkcja straty

Dla embeddingu $x \in \mathbb{R}^{512}$ nasz SAE wylicza nieujemne aktywacje $a \in \mathbb{R}^{512}_{\ge 0}$ oraz rekonstrukcję $\hat{x}$:

$$a = \text{ReLU}\big(W_{\text{enc}}(x - b_{\text{tied}}) + b_{\text{enc}}\big), \qquad \hat{x} = a W_{\text{dec}}^T + b_{\text{tied}}$$

Elementy implementacji:
1. Parametr `tied_bias` ($b_{\text{tied}}$) odejmujemy przed enkoderem i dodajemy po dekoderze. Gdy wszystkie aktywacje wynoszą zero, rekonstrukcja wynosi $b_{\text{tied}}$.
2. Kolumny dekodera utrzymujemy na długości 1 przez `_remove_parallel_gradient` oraz `normalize_decoder()` po każdym kroku Adama. Bez tego ograniczenia sieć mogłaby wydłużać kolumny dekodera i zmniejszać aktywacje $a$, co obniża karę L1 bez zmiany rekonstrukcji i bez zerowania neuronów SAE.
3. Funkcja straty łączy błąd kwadratowy MSE i karę L1 ($\lambda \sum_j |a_j|$). Liczbę aktywnych neuronów na obrazie mierzy `avg_l0`. Optymalizujemy L1 zamiast L0, ponieważ pochodna funkcji schodkowej L0 wynosi zero poza punktem zero.

Po pierwszym uruchomieniu możesz zmienić `L1_COEFFICIENT` na `0.002` albo `0.02`, albo zakomentować `sae.normalize_decoder()`, i sprawdzić wpływ tej zmiany na `avg_l0` i `recovered_accuracy`.


In [ ]:
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim, n_neurons):
        super().__init__()
        self.input_dim = input_dim
        self.n_neurons = n_neurons
        self.tied_bias = nn.Parameter(torch.zeros(input_dim))
        self.encoder = nn.Linear(input_dim, n_neurons)
        self.decoder_weight = nn.Parameter(torch.empty(input_dim, n_neurons))
        self.reset_parameters()
        self.decoder_weight.register_hook(self._remove_parallel_gradient)

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.encoder.weight, nonlinearity="relu")
        bound = 1 / math.sqrt(self.input_dim)
        nn.init.uniform_(self.encoder.bias, -bound, bound)
        nn.init.normal_(self.decoder_weight)
        self.normalize_decoder()

    @torch.no_grad()
    def normalize_decoder(self):
        """Rzutuje kolumny dekodera na sferę jednostkową po kroku Adama."""
        F.normalize(self.decoder_weight, dim=0, out=self.decoder_weight)

    def _remove_parallel_gradient(self, grad):
        """Usuwa składową gradientu równoległą do kolumn dekodera."""
        unit = self.decoder_weight / self.decoder_weight.norm(dim=0, keepdim=True)
        return grad - unit * (grad * unit).sum(dim=0, keepdim=True)

    def encode(self, x):
        return F.relu(self.encoder(x - self.tied_bias))

    def decode(self, activations):
        return activations @ self.decoder_weight.T + self.tied_bias

    def forward(self, x):
        activations = self.encode(x)
        return activations, self.decode(activations)


def sae_loss(x, activations, reconstructed, l1_coefficient):
    reconstruction = (reconstructed - x).pow(2).mean(dim=-1).mean()
    l1_penalty = l1_coefficient * activations.abs().sum(dim=-1).mean()
    avg_l0 = (activations > 0).float().sum(dim=-1).mean()
    return {
        "total_loss": reconstruction + l1_penalty,
        "reconstruction_loss": reconstruction,
        "l1_penalty": l1_penalty,
        "avg_l0": avg_l0,
    }


## Krok 4. Trening Sparse Autoencodera

Trenujemy SAE na macierzy `train_features` o kształcie `(5394, 512)` przechowywanej w pamięci urządzenia. Early stopping zatrzymuje trening, gdy `val total_loss` nie poprawia się przez 50 epok.


In [ ]:
def run_epoch(sae, features, batch_size, l1_coefficient, optimizer=None, generator=None):
    train = optimizer is not None
    sae.train(train)
    order = torch.randperm(len(features), generator=generator).to(features.device) if train else None
    totals = {"reconstruction_loss": 0.0, "l1_penalty": 0.0, "total_loss": 0.0, "avg_l0": 0.0}

    with torch.set_grad_enabled(train):
        for start in range(0, len(features), batch_size):
            batch = features[order[start:start + batch_size]] if train else features[start:start + batch_size]
            losses = sae_loss(batch, *sae(batch), l1_coefficient)
            if train:
                optimizer.zero_grad()
                losses["total_loss"].backward()
                optimizer.step()
                sae.normalize_decoder()
            for key in totals:
                totals[key] += losses[key].item() * len(batch)

    return {key: value / len(features) for key, value in totals.items()}


set_seed(SEED)
tr_x = train_features.to(device)
va_x = val_features.to(device)
va_y = val_labels.long().to(device)

input_dim = tr_x.shape[1]
n_neurons = int(input_dim * EXPANSION_FACTOR)
sae = SparseAutoencoder(input_dim, n_neurons).to(device)
optimizer = torch.optim.Adam(sae.parameters(), lr=SAE_LR)
generator = torch.Generator().manual_seed(SEED)

history, best_state, best_epoch, best_val_loss, stale = [], None, 0, float("inf"), 0

pbar = tqdm(range(1, SAE_EPOCHS + 1), desc="Trening SAE")
for epoch in pbar:
    tr_m = run_epoch(sae, tr_x, SAE_BATCH_SIZE, L1_COEFFICIENT, optimizer, generator)
    va_m = run_epoch(sae, va_x, SAE_BATCH_SIZE, L1_COEFFICIENT)
    history.append({"epoch": epoch, **{f"train_{k}": v for k, v in tr_m.items()}, **{f"val_{k}": v for k, v in va_m.items()}})

    if va_m["total_loss"] < best_val_loss:
        best_val_loss, best_epoch, stale = va_m["total_loss"], epoch, 0
        best_state = copy.deepcopy(sae.state_dict())
    else:
        stale += 1

    if epoch % 10 == 0 or stale >= SAE_PATIENCE:
        pbar.set_postfix(val_mse=f"{va_m['reconstruction_loss']:.4f}", train_L0=f"{tr_m['avg_l0']:.1f}", best=best_epoch)
    if stale >= SAE_PATIENCE:
        print(f"Early stopping w epoce {epoch}. Najlepsza epoka: {best_epoch}.")
        break

sae.load_state_dict(best_state)
sae.eval()
final_train = run_epoch(sae, tr_x, SAE_BATCH_SIZE, L1_COEFFICIENT)
final_val = run_epoch(sae, va_x, SAE_BATCH_SIZE, L1_COEFFICIENT)

print(f"Najlepsza epoka: {best_epoch}.")
print(f"Train: MSE = {final_train['reconstruction_loss']:.4f}, avg L0 = {final_train['avg_l0']:.2f}.")
print(f"Val:   MSE = {final_val['reconstruction_loss']:.4f}, avg L0 = {final_val['avg_l0']:.2f}.")

epochs_x = [r["epoch"] for r in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric, title in zip(axes, ["reconstruction_loss", "avg_l0"], ["Reconstruction loss (MSE)", "Average L0 (aktywne neurony na obraz)"]):
    ax.plot(epochs_x, [r[f"train_{metric}"] for r in history], label="Train")
    ax.plot(epochs_x, [r[f"val_{metric}"] for r in history], label="Validation")
    ax.axvline(best_epoch, color="gray", linestyle="--", alpha=0.6, label=f"Best ({best_epoch})")
    ax.set_xlabel("Epoka")
    ax.set_title(title)
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.show()


### Porównanie dla tego samego zdjęcia: obraz, embedding ResNet-18 i embedding SAE

Poniższy wykres zestawia dla zdjęcia `SAMPLE_IDX` jego surowy embedding z ResNet-18 oraz wektor aktywacji SAE. Sprawdzamy też, jaką klasę przewiduje głowica `backbone.head` na podstawie oryginalnego embeddingu i na podstawie rekonstrukcji z SAE.


In [ ]:
with torch.no_grad():
    x_one = train_features[SAMPLE_IDX : SAMPLE_IDX + 1].to(device)
    sae_act_one, x_recon_one = sae(x_one)
    pred_orig = train_full.class_names[int(backbone.head(x_one).argmax(1).item())]
    pred_recon = train_full.class_names[int(backbone.head(x_recon_one).argmax(1).item())]

sample_sae_emb = sae_act_one[0].cpu().numpy()
active_sae_idx = np.where(sample_sae_emb > 0)[0]
active_sae_sorted = active_sae_idx[np.argsort(-sample_sae_emb[active_sae_idx])]
n_active_sae = len(active_sae_idx)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.0), gridspec_kw={"width_ratios": [1, 1.9, 1.9]})

axes[0].imshow(display_transform(sample_img))
axes[0].set_title(f"1. Zdjęcie #{SAMPLE_IDX}\nKlasa: {sample_label}", fontsize=10)
axes[0].axis("off")

axes[1].bar(np.arange(len(sample_resnet_emb)), sample_resnet_emb, width=1.0, color="#1d4e89")
axes[1].set_xlim(0, len(sample_resnet_emb))
axes[1].set_xlabel("Indeks wymiaru (0 do 511)")
axes[1].set_ylabel("Wartość aktywacji")
axes[1].set_title(f"2. Embedding ResNet-18\nAktywnych: {n_active_resnet} z 512. Predykcja: {pred_orig}", fontsize=10)
axes[1].grid(True, axis="y", alpha=0.4)

axes[2].bar(np.arange(len(sample_sae_emb)), sample_sae_emb, width=2.5, color="#8b2635")
axes[2].set_xlim(0, len(sample_sae_emb))
if sample_sae_emb.max() > 0:
    axes[2].set_ylim(0, sample_sae_emb.max() * 1.18)
axes[2].set_xlabel("Indeks neuronu SAE (0 do 511)")
axes[2].set_ylabel("Wartość aktywacji SAE")
axes[2].set_title(f"3. Embedding SAE\nAktywnych (L0): {n_active_sae} z 512. Predykcja z rekonstr.: {pred_recon}", fontsize=10)
axes[2].grid(True, axis="y", alpha=0.4)

for n_id in active_sae_sorted[:3]:
    axes[2].annotate(
        f"n{n_id}",
        xy=(n_id, sample_sae_emb[n_id]),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        fontsize=8,
        fontweight="bold",
        color="#8b2635",
    )

plt.tight_layout()
plt.show()

print(f"Liczba aktywnych jednostek dla zdjęcia #{SAMPLE_IDX} spadła z {n_active_resnet} z 512 w ResNet-18 do {n_active_sae} z 512 w SAE.")
print("Aktywne neurony SAE posortowane malejąco:")
for n_id in active_sae_sorted[:10]:
    print(f"  neuron {n_id:>3}: aktywacja = {sample_sae_emb[n_id]:.3f}")


## Krok 5. Wierność klasyfikatora i pruning neuronów

Błąd MSE nie mówi, czy zrekonstruowane embeddingi pozwalają głowicy `backbone.head` poprawnie rozpoznawać gatunki. Sprawdzamy to na zbiorze `val` za pomocą dwóch metryk:
- `recovered_loss` to ułamek straty Cross-Entropy odzyskany po podaniu rekonstrukcji SAE zamiast oryginalnych embeddingów. Zero oznacza wynik równy wektorowi samych zer, czyli zero-ablację, a 100% oznacza wynik równy oryginalnym embeddingom.
- `recovered_accuracy` działa analogicznie dla dokładności klasyfikatora.

Następnie usuwamy martwe i rzadko aktywne neurony SAE. Algorytm `run_pruning` wybiera najwyższy próg liczby aktywnych obrazów, co najmniej 25, dla którego spadek `recovered_loss` na zbiorze `val` nie przekracza 0.005.


In [ ]:
@torch.no_grad()
def recovered_metrics(sae, head, features, labels, kept_mask):
    def ce_and_acc(inputs):
        logits = head(inputs)
        return F.cross_entropy(logits, labels).item(), (logits.argmax(1) == labels).float().mean().item()

    orig_loss, orig_acc = ce_and_acc(features)
    zero_loss, zero_acc = ce_and_acc(torch.zeros_like(features))
    recon = sae.decode(sae.encode(features) * kept_mask.to(features.device))
    rec_loss, rec_acc = ce_and_acc(recon)

    recovered_loss = (zero_loss - rec_loss) / (zero_loss - orig_loss + 1e-8)
    recovered_acc = (rec_acc - zero_acc) / (orig_acc - zero_acc + 1e-8)
    return recovered_loss, recovered_acc


@torch.no_grad()
def run_pruning(sae, head, train_counts, val_features, val_labels, tolerance, min_active_images):
    total = train_counts.numel()
    base_loss, base_acc = recovered_metrics(sae, head, val_features, val_labels, torch.ones(total, dtype=torch.bool))

    min_th = int(min_active_images) - 1
    max_cnt = int(train_counts.max())
    thresholds = sorted({min_th} | {c for c in train_counts.unique().tolist() if min_th <= c < max_cnt})

    curve, selected, post_idx = [], None, None
    for i, th in enumerate(thresholds):
        if post_idx is not None and i not in post_idx:
            continue
        mask = train_counts > th
        if not mask.any():
            continue
        r_loss, r_acc = recovered_metrics(sae, head, val_features, val_labels, mask)
        drop = base_loss - r_loss
        if post_idx is None and drop <= tolerance:
            selected = {"threshold": th, "kept_mask": mask, "recovered_loss": r_loss, "recovered_accuracy": r_acc, "loss_drop": drop}
        elif post_idx is None:
            rem = list(range(i + 1, len(thresholds)))
            if len(rem) > 100:
                rem = [rem[k] for k in np.linspace(0, len(rem) - 1, 100, dtype=int)]
            post_idx = set(rem)
        curve.append({"threshold": th, "kept_neurons": int(mask.sum()), "recovered_loss": r_loss, "recovered_accuracy": r_acc, "loss_drop": drop})

    if selected is None:
        mask = train_counts > min_th
        r_loss, r_acc = recovered_metrics(sae, head, val_features, val_labels, mask)
        selected = {"threshold": min_th, "kept_mask": mask, "recovered_loss": r_loss, "recovered_accuracy": r_acc, "loss_drop": base_loss - r_loss}

    kept_indices = torch.nonzero(selected["kept_mask"]).squeeze(1).tolist()
    summary = {
        "total_neurons": total,
        "dead_neurons": int((train_counts == 0).sum()),
        "kept_neurons": len(kept_indices),
        "selected_threshold": selected["threshold"],
        "baseline_recovered_loss": base_loss,
        "baseline_recovered_accuracy": base_acc,
        "selected_recovered_loss": selected["recovered_loss"],
        "selected_recovered_accuracy": selected["recovered_accuracy"],
        "recovered_loss_drop": selected["loss_drop"],
    }
    return kept_indices, curve, summary


with torch.no_grad():
    train_sae_activations = sae.encode(tr_x).cpu()
neuron_counts = (train_sae_activations > 0).sum(dim=0)

kept_indices, pruning_curve, pruning_summary = run_pruning(
    sae, backbone.head, neuron_counts, va_x, va_y, CE_DROP_TOLERANCE, MIN_ACTIVE_IMAGES
)

print(f"Martwe neurony (0 zdjęć):  {pruning_summary['dead_neurons']} z {pruning_summary['total_neurons']}.")
print(f"Zachowane po pruningu:     {pruning_summary['kept_neurons']} z {pruning_summary['total_neurons']} (próg powyżej {pruning_summary['selected_threshold']} zdjęć).")
print(f"Recovered CE Loss:         przed pruningiem {pruning_summary['baseline_recovered_loss']:.2%}, po pruningu {pruning_summary['selected_recovered_loss']:.2%}.")
print(f"Recovered Accuracy:        przed pruningiem {pruning_summary['baseline_recovered_accuracy']:.2%}, po pruningu {pruning_summary['selected_recovered_accuracy']:.2%}.")

counts_np = neuron_counts.numpy()
log_dens = np.log10(np.clip(counts_np / len(train_features), 1e-8, None))
th_sel = pruning_summary["selected_threshold"]
dead = counts_np == 0
low = (counts_np <= th_sel) & ~dead
high = counts_np > th_sel

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist([log_dens[dead], log_dens[low], log_dens[high]], bins=60, stacked=True,
             color=["purple", "orange", "green"], edgecolor="black", label=["Martwe", "Przycięte", "Zachowane"])
axes[0].set_xlabel("log10(gęstość aktywacji)")
axes[0].set_ylabel("Liczba neuronów SAE")
axes[0].set_title("Gęstość cech na zbiorze train")
axes[0].legend()
axes[0].grid(True)

ths = [r["threshold"] for r in pruning_curve]
for ax, key, title in zip(axes[1:], ["recovered_loss", "kept_neurons"], ["Recovered CE Loss", "Zachowane neurony SAE"]):
    vals = [r[key] for r in pruning_curve]
    ax.plot(ths, vals)
    if th_sel in ths:
        ax.scatter([th_sel], [vals[ths.index(th_sel)]], color="red", marker="*", s=140, zorder=5, label=f"Próg = {th_sel}")
        ax.legend()
    ax.set_xlabel("Próg liczby aktywnych obrazów")
    ax.set_title(title)
    ax.grid(True)

plt.tight_layout()
plt.show()


## Krok 6. Galeria Top-K dla zachowanych neuronów SAE

Dla wybranego neuronu SAE wyświetlamy `K` zdjęć treningowych o najwyższej aktywacji oraz histogram najczęstszych gatunków wśród Top-50 zdjęć. Histogram pomaga odróżnić wspólną cechę wizualną, gdzie aktywacje rozkładają się na wiele gatunków, od detektora jednej klasy.

Przejrzyj kilka neuronów SAE w widgecie poniżej. Znajdź jeden neuron odpowiadający pojedynczemu gatunkowi oraz jeden neuron obejmujący wspólną cechę wizualną, na przykład kolor, tło albo kształt dzioba.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

IMAGES_PER_ROW = 5


class NeuronExplorer:
    def __init__(self, activations, labels, image_names, kept_indices, neuron_counts, dataset):
        self.activations = activations
        self.labels = labels
        self.image_names = image_names
        self.kept_indices = kept_indices
        self.neuron_counts = neuron_counts
        self.dataset = dataset
        self.class_names = dataset.class_names

    def neurons_by_frequency(self):
        return sorted(self.kept_indices, key=lambda n: -int(self.neuron_counts[n]))

    def top_images(self, neuron, k):
        values, indices = self.activations[:, neuron].topk(k)
        pos = values > 0
        return indices[pos].tolist(), values[pos].tolist()

    def plot_neuron(self, neuron, k=10, hist_top=50, hist_classes=10):
        indices, values = self.top_images(neuron, k)
        shown = [display_transform(self.dataset.load_image_by_name(self.image_names[i])) for i in indices]

        rows = math.ceil(len(shown) / IMAGES_PER_ROW)
        fig = plt.figure(figsize=(3 * IMAGES_PER_ROW, 3.3 * rows + 2.8))
        grid = fig.add_gridspec(rows + 1, IMAGES_PER_ROW, height_ratios=[3.3] * rows + [2.5])
        for pos, (img, idx, val) in enumerate(zip(shown, indices, values)):
            ax = fig.add_subplot(grid[pos // IMAGES_PER_ROW, pos % IMAGES_PER_ROW])
            ax.imshow(img)
            ax.set_title(f"{textwrap.fill(self.class_names[int(self.labels[idx])], 22)}\nact = {val:.2f}", fontsize=9)
            ax.axis("off")

        top_idx, _ = self.top_images(neuron, hist_top)
        counts = Counter(self.class_names[int(self.labels[i])] for i in top_idx).most_common(hist_classes)
        hax = fig.add_subplot(grid[rows, :])
        hax.barh([c[0] for c in reversed(counts)], [c[1] for c in reversed(counts)])
        hax.set_title(f"Najczęstsze gatunki wśród top {len(top_idx)} aktywujących zdjęć", fontsize=10)
        hax.set_xlabel("Liczba zdjęć")

        fires = int(self.neuron_counts[neuron])
        fig.suptitle(f"Neuron SAE {neuron}: aktywny na {fires} z {len(self.image_names)} zdjęć ({fires / len(self.image_names):.1%})", fontsize=13)
        fig.tight_layout()
        return fig


explorer = NeuronExplorer(
    train_sae_activations, train_labels, train_image_names, kept_indices, neuron_counts, train_full
)

picker = widgets.Dropdown(
    options=[(f"neuron {u} (aktywny na {int(neuron_counts[u])} zdjęciach)", u) for u in explorer.neurons_by_frequency()],
    description="Neuron:",
    layout=widgets.Layout(width="420px"),
)
k_slider = widgets.IntSlider(value=10, min=5, max=20, step=5, description="Top-K:")


def show_neuron(neuron, k):
    explorer.plot_neuron(neuron, k=k)
    plt.show()


out = widgets.interactive_output(show_neuron, {"neuron": picker, "k": k_slider})
display(widgets.HBox([picker, k_slider]), out)


## Krok 7. Czystość klasowa i wariancja wewnątrzklasowa

Poniższa komórka mierzy, ile zachowanych neuronów SAE przypomina pojedyncze gatunki, a ile obejmuje różne klasy:
- Detektor klasy (`class_detector`) to neuron SAE, dla którego jeden gatunek stanowi co najmniej 80% zdjęć w Top-10.
- Neuron rozproszony (`diffuse`) to neuron SAE, którego Top-10 obejmuje co najmniej 8 różnych gatunków.
- Wariancja wewnątrzklasowa (`within_class_variance`) to ułamek całkowitej wariancji embeddingów ResNet-18 pozostający wewnątrz klas. Niska wartość oznacza silne skupienie klas wokół ich średnich. To zjawisko Neural Collapse opisane przez Papyan, Han i Donoho (PNAS 2020), które ułatwia SAE kodowanie jednego neuronu na jedną klasę.

Na końcu wypisujemy po 5 indeksów neuronów z obu grup, które można wybrać w widgecie w Kroku 6.


In [ ]:
def neuron_purity(activations, labels, neuron, k):
    values, indices = activations[:, neuron].topk(k)
    counts = Counter(labels[indices[values > 0]].tolist())
    top_class, top_count = counts.most_common(1)[0]
    return {"neuron": neuron, "top_class": top_class, "share": top_count / counts.total(), "n_classes": len(counts)}


def within_class_variance_share(features, labels):
    within = sum((features[labels == c] - features[labels == c].mean(0)).pow(2).sum() for c in labels.unique())
    return (within / (features - features.mean(0)).pow(2).sum()).item()


purity_rows = [neuron_purity(train_sae_activations, train_labels, n, k=10) for n in kept_indices]
wide_shares = [neuron_purity(train_sae_activations, train_labels, n, k=50)["share"] for n in kept_indices]
detectors = [r for r in purity_rows if r["share"] >= 0.8]
diffuse = [r for r in purity_rows if r["n_classes"] >= 8]
wcv = within_class_variance_share(train_features, train_labels)

print(f"Zachowane neurony SAE:            {len(kept_indices)}")
print(f"Detektory klas (>=80% w Top-10):  {len(detectors) / len(kept_indices):.1%} ({len(detectors)} neuronów, {len({r['top_class'] for r in detectors})} gatunków)")
print(f"Neurony rozproszone (>=8 klas):   {len(diffuse) / len(kept_indices):.1%} ({len(diffuse)} neuronów)")
print(f"Średni udział głównej klasy:      Top-10 = {np.mean([r['share'] for r in purity_rows]):.2f}, Top-50 = {np.mean(wide_shares):.2f}")
print(f"Within-class variance (ResNet):   {wcv:.1%}")

plt.figure(figsize=(8, 3.8))
plt.hist([r["share"] for r in purity_rows], bins=[(i + 0.5) / 10 for i in range(11)], edgecolor="black")
plt.xlabel("Udział najczęstszej klasy w Top-10")
plt.ylabel("Liczba zachowanych neuronów SAE")
plt.title("Rozkład czystości klasowej zachowanych neuronów SAE")
plt.grid(True)
plt.show()

most_shared = sorted(purity_rows, key=lambda r: (r["share"], -r["n_classes"]))[:5]
most_specific = sorted(purity_rows, key=lambda r: (-r["share"], r["n_classes"]))[:5]
print("\nNeurony z najniższym udziałem jednej klasy w Top-10:")
for r in most_shared:
    print(f"  neuron {r['neuron']:>3}: główna klasa {r['share']:.0%} ({train_full.class_names[r['top_class']]}), liczba klas w Top-10 = {r['n_classes']}")
print("\nNeurony z najwyższym udziałem jednej klasy w Top-10:")
for r in most_specific:
    print(f"  neuron {r['neuron']:>3}: główna klasa {r['share']:.0%} ({train_full.class_names[r['top_class']]}), liczba klas w Top-10 = {r['n_classes']}")
